In [ ]:
"""
ASL Static Letter Classification — Custom CNN (PyTorch)
========================================================
Dataset  : /kaggle/input/datasets/ep25b021/dataset-1/asl_processed
Structure: train/{A-Z}/  →  800 images/class
           test/{A-Z}/   →  200 images/class
Classes  : 26 (A–Z)
"""

import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

# ─────────────────────────────────────────────
# 0.  Reproducibility
# ─────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

# ─────────────────────────────────────────────
# 1.  Config
# ─────────────────────────────────────────────
DATA_ROOT   = Path("/kaggle/input/datasets/ep25b021/dataset-1/asl_processed")
TRAIN_DIR   = DATA_ROOT / "train"
TEST_DIR    = DATA_ROOT / "test"
SAVE_PATH   = Path("asl_cnn_weights.pth")

IMG_SIZE    = 64        # resize all images to 64×64
BATCH_SIZE  = 64
VAL_SPLIT   = 0.15      # 15 % of train → validation
NUM_CLASSES = 26
EPOCHS      = 40
LR          = 3e-4
WEIGHT_DECAY= 1e-4
PATIENCE    = 8         # early-stopping patience

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ─────────────────────────────────────────────
# 2.  Data Transforms
# ─────────────────────────────────────────────
# Training  → aggressive augmentation to fight overfitting
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])

# Validation / Test  → deterministic
eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std= [0.229, 0.224, 0.225]),
])


# ─────────────────────────────────────────────
# 3.  Datasets & DataLoaders
# ─────────────────────────────────────────────
full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset       = datasets.ImageFolder(TEST_DIR,  transform=eval_transforms)

# Train / Val split (stratified-ish via random_split — balanced dataset so fine)
n_total = len(full_train_dataset)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val
train_dataset, val_dataset = random_split(full_train_dataset, [n_train, n_val],
                                          generator=torch.Generator().manual_seed(SEED))

# Validation should use eval transforms — wrap with a thin dataset class
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        # img already transformed by parent; re-load via base dataset
        # Access the underlying dataset and index
        base_idx = self.subset.indices[idx]
        path, label = self.subset.dataset.samples[base_idx]
        from PIL import Image as PILImage
        img = PILImage.open(path).convert("RGB")
        img = self.transform(img)
        return img, label

val_dataset_eval = TransformSubset(val_dataset, eval_transforms)

train_loader = DataLoader(train_dataset,     batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset_eval,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,      batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

CLASS_NAMES = full_train_dataset.classes   # ['A','B',...,'Z']
print(f"Classes : {CLASS_NAMES}")
print(f"Train   : {n_train}  |  Val: {n_val}  |  Test: {len(test_dataset)}")


# ─────────────────────────────────────────────
# 4.  Custom CNN Architecture
# ─────────────────────────────────────────────
# Design rationale
# ───────────────
#  • Four progressively deeper conv-blocks with BatchNorm → stable training.
#  • Residual-style skip connections inside each block (like a lite ResNet)
#    without importing torchvision models — coded fully from scratch.
#  • Spatial Pyramid Pooling before the classifier → multi-scale features.
#  • Dropout on FC layers → regularisation.
#  • Kernel sizes: 3×3 throughout (best compute/capacity trade-off at 64px input).

class ConvBnRelu(nn.Module):
    """Conv → BN → ReLU building block."""
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel,
                      stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock(nn.Module):
    """
    Two ConvBnRelu layers with a residual skip.
    If channel dimensions differ, a 1×1 conv aligns them.
    """
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = ConvBnRelu(in_ch,  out_ch, stride=stride)
        self.conv2 = ConvBnRelu(out_ch, out_ch, stride=1)

        self.skip = nn.Sequential()
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        return nn.functional.relu(out + self.skip(x), inplace=True)


class SpatialPyramidPooling(nn.Module):
    """
    SPP with fixed output size regardless of spatial input size.
    Bins: 1×1, 2×2, 4×4  → concatenated along channel dim.
    """
    def __init__(self, levels=(1, 2, 4)):
        super().__init__()
        self.levels = levels

    def forward(self, x):
        B, C, H, W = x.shape
        out = []
        for lvl in self.levels:
            pool = nn.functional.adaptive_avg_pool2d(x, output_size=lvl)
            out.append(pool.view(B, -1))          # flatten spatial dims
        return torch.cat(out, dim=1)


class ASLCNN(nn.Module):
    """
    Custom CNN for 26-class ASL letter recognition.

    Input : (B, 3, 64, 64)
    Output: (B, 26) logits

    Architecture overview
    ─────────────────────
    Stem      : 3→32, 3×3 conv
    Block 1   : ResidualBlock  32→64   + MaxPool  → 32×32
    Block 2   : ResidualBlock  64→128  + MaxPool  → 16×16
    Block 3   : ResidualBlock 128→256  + MaxPool  →  8×8
    Block 4   : ResidualBlock 256→512  + MaxPool  →  4×4
    SPP       : levels (1,2,4) → flattened 512*(1+4+16)=10,752-d
    FC head   : 10752 → 1024 → 512 → 26
    """
    def __init__(self, num_classes=26, dropout=0.4):
        super().__init__()

        # Stem
        self.stem = ConvBnRelu(3, 32, kernel=3, padding=1)

        # Four residual blocks + pooling
        self.block1 = nn.Sequential(
            ResidualBlock(32,  64),
            nn.MaxPool2d(2, 2),       # 64→32
            nn.Dropout2d(0.1),
        )
        self.block2 = nn.Sequential(
            ResidualBlock(64,  128),
            nn.MaxPool2d(2, 2),       # 32→16
            nn.Dropout2d(0.15),
        )
        self.block3 = nn.Sequential(
            ResidualBlock(128, 256),
            nn.MaxPool2d(2, 2),       # 16→8
            nn.Dropout2d(0.2),
        )
        self.block4 = nn.Sequential(
            ResidualBlock(256, 512),
            nn.MaxPool2d(2, 2),       # 8→4
            nn.Dropout2d(0.2),
        )

        # Spatial Pyramid Pooling
        self.spp = SpatialPyramidPooling(levels=(1, 2, 4))
        # 512 channels × (1² + 2² + 4²) = 512 × 21 = 10,752
        spp_out_dim = 512 * (1 + 4 + 16)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(spp_out_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout / 2),

            nn.Linear(512, num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)     # (B, 32,  64, 64)
        x = self.block1(x)   # (B, 64,  32, 32)
        x = self.block2(x)   # (B, 128, 16, 16)
        x = self.block3(x)   # (B, 256,  8,  8)
        x = self.block4(x)   # (B, 512,  4,  4)
        x = self.spp(x)      # (B, 10752)
        x = self.classifier(x)
        return x


# ─── Sanity-check the architecture ─────────────────────────────────────────
model = ASLCNN(num_classes=NUM_CLASSES).to(DEVICE)
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out   = model(dummy)
assert out.shape == (2, NUM_CLASSES), f"Unexpected output shape: {out.shape}"

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters : {total_params:,}")
print(f"Output shape     : {out.shape}")


# ─────────────────────────────────────────────
# 5.  Loss, Optimiser, Scheduler
# ─────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)   # label smoothing → generalisation
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)


# ─────────────────────────────────────────────
# 6.  Training Loop
# ─────────────────────────────────────────────
def run_epoch(loader, model, criterion, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            n          += imgs.size(0)

    return total_loss / n, correct / n


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc   = 0.0
best_weights   = None
no_improve     = 0

print("\n{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  "
      "{'Val Loss':>8}  {'Val Acc':>7}  {'LR':>9}")
print("─" * 65)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   model, criterion,            train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    lr_now = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0
    print(f"{epoch:5d}  {tr_loss:10.4f}  {tr_acc*100:8.2f}%  "
          f"{vl_loss:8.4f}  {vl_acc*100:6.2f}%  {lr_now:.2e}  ({elapsed:.1f}s)")

    # ── Early stopping + checkpoint ─────────────────────
    if vl_acc > best_val_acc:
        best_val_acc  = vl_acc
        best_weights  = copy.deepcopy(model.state_dict())
        no_improve    = 0
        print(f"          ✓ New best val acc: {best_val_acc*100:.2f}%  → saving weights")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs "
                  f"(no improvement for {PATIENCE} consecutive epochs).")
            break

# ── Restore best weights ────────────────────────────────
model.load_state_dict(best_weights)
torch.save({
    "model_state_dict" : best_weights,
    "class_names"      : CLASS_NAMES,
    "img_size"         : IMG_SIZE,
    "num_classes"      : NUM_CLASSES,
    "best_val_acc"     : best_val_acc,
}, SAVE_PATH)
print(f"\nWeights saved → {SAVE_PATH}  (best val acc = {best_val_acc*100:.2f}%)")


# ─────────────────────────────────────────────
# 7.  Evaluation on Test Set
# ─────────────────────────────────────────────
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"\nTest Accuracy : {test_acc * 100:.2f}%")
print("\nPer-class report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))


# ─────────────────────────────────────────────
# 8.  Plots
# ─────────────────────────────────────────────
epochs_ran = len(history["train_loss"])
x          = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(x, history["train_loss"], label="Train Loss",  color="#E07B54")
axes[0].plot(x, history["val_loss"],   label="Val Loss",    color="#5B8DB8", linestyle="--")
axes[0].set_title("Loss over Epochs");  axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss");             axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(x, [a*100 for a in history["train_acc"]], label="Train Acc", color="#E07B54")
axes[1].plot(x, [a*100 for a in history["val_acc"]],   label="Val Acc",   color="#5B8DB8",
             linestyle="--")
axes[1].set_title("Accuracy over Epochs"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)");        axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("ASL CNN — Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f"Confusion Matrix — Test Set  (Acc = {test_acc*100:.2f}%)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nDone. Artefacts saved:")
print("  asl_cnn_weights.pth")
print("  training_curves.png")
print("  confusion_matrix.png")